Create company brochure using Gradio (interface) 

In [32]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents

In [33]:
import gradio as gr # oh yeah!

In [34]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AQ.Ab8RN


In [35]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [ ]:
# def get_links_user_prompt(url):
#     user_prompt = f"""
# Here is the list of links on the website {url} -
# Please decide which of these are relevant web links for a brochure about the company, 
# respond with the full https URL in JSON format.
# Do not include Terms of Service, Privacy, email links.

# Links (some might be relative links):

# """
#     links = fetch_website_links(url)
#     user_prompt += "\n".join(links)
#     return user_prompt

In [ ]:
# link_system_prompt = """
# You are provided with a list of links found on a webpage.
# You are able to decide which of the links would be most relevant to include in a brochure about the company,
# such as links to an About page, or a Company page, or Careers/Jobs pages.
# You should respond in JSON as in this example:

# {
#     "links": [
#         {"type": "about page", "url": "https://full.url/goes/here/about"},
#         {"type": "careers page", "url": "https://another.full.url/careers"}
#     ]
# }
# """

In [ ]:
# def select_relevant_links(url):
#     response = gemini.chat.completions.create(
#         model='gemini-3.1-flash-lite',
#         messages=[
#             {"role": "system", "content": link_system_prompt},
#             {"role": "user", "content": get_links_user_prompt(url)}
#         ],
#         response_format={"type": "json_object"}
#     )
#     result = response.choices[0].message.content
#     links = json.loads(result)
#     return links

In [ ]:
# def fetch_page_and_all_relevant_links(url):
#     contents = fetch_website_contents(url)
#     relevant_links = select_relevant_links(url)
#     result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
#     for link in relevant_links['links']:
#         result += f"\n\n### Link: {link['type']}\n"
#         result += fetch_website_contents(link["url"])
#     return result

In [36]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [37]:
# Let's create a call that streams back results
# If you'd like a refresher on Generators (the "yield" keyword),
# Please take a look at the Intermediate Python guide in the guides folder

def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = gemini.chat.completions.create(
        model= 'gemini-3.1-flash-lite',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [38]:
def stream_ollama(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = ollama.chat.completions.create(
        model= 'llama3.2',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [40]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="Gemini":
        result = stream_gemini(prompt)
    elif model=="Ollama":
        result = stream_ollama(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input_company = gr.Textbox(label="Company Name:", info="Enter a company name: ")
message_input_url = gr.Textbox(label="Company URL:", info="Enter a company URL: ")
model_selector = gr.Dropdown(["Gemini", "Ollama"], label="Select model", value="Gemini")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[message_input_company, message_input_url, model_selector], 
    outputs=[message_output], 
    examples=[
        ["HuggingFace", "https://huggingface.co", "Gemini"],
        ["Tap Payments", "https://www.tap.company/", "Ollama"],
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.
